In [10]:
# Stage 3: Reranking

#The reranking stage takes the Top 20 chunks retrieved during the chunking/retrieval stage and reorders them based on their relevance to the user's question.

#The goal is to identify the 5 most useful chunks to pass to the LLM.

#This notebook implements two reranking methods:
#- No reranker 
#- Cross-Encoder
#- LLM Reranker
#- Flash reranking


In [11]:
%pip -q install datasets pandas matplotlib pyarrow sentence-transformers faiss-cpu requests
     

Note: you may need to restart the kernel to use updated packages.


In [12]:

# Import libraries

import time
import json
import re

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import faiss 

from collections import defaultdict
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder

SEED = 42
np.random.seed(SEED)

print("Imports successful.")



Imports successful.


In [16]:
DATASET_ID = "allenai/qasper"
EVAL_SPLIT = "validation"


MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

RERANK_TOP_K = 20
TOP_N = 5

# Set to an integer (for example, 5) to run a small test first.
# Keep None to evaluate every question with textual evidence.
EVAL_QUESTION_LIMIT = None

print("Dataset:", DATASET_ID)
print("Split:", EVAL_SPLIT)


Dataset: allenai/qasper
Split: validation


In [17]:
response = requests.get(
    "https://datasets-server.huggingface.co/parquet",
    params={"dataset": DATASET_ID},
    timeout=60,
)
response.raise_for_status()
parquet_info = response.json()

configs = list(dict.fromkeys(
    item["config"] for item in parquet_info["parquet_files"]
))
preferred_config = "qasper" if "qasper" in configs else configs[0]

data_files = {}
for split_name in ["train", "validation", "test"]:
    urls = [
        item["url"]
        for item in parquet_info["parquet_files"]
        if item["config"] == preferred_config
        and item["split"] == split_name
    ]
    if urls:
        data_files[split_name] = urls

if EVAL_SPLIT not in data_files:
    raise ValueError(f"Could not find split: {EVAL_SPLIT}")

dataset = load_dataset("parquet", data_files=data_files)
split = dataset[EVAL_SPLIT]

n_papers = min(
    MAX_PAPERS if MAX_PAPERS is not None else len(split),
    len(split),
)
eval_papers = [split[i] for i in range(n_papers)]

print("Dataset split size:", len(split))
print("Papers selected:", len(eval_papers))

Dataset split size: 281
Papers selected: 30


In [18]:
def extract_evidence(paper, question_index):
    record = paper["qas"]["answers"][question_index]
    evidence_list = []

    for annotation in record["answer"]:
        if annotation["unanswerable"]:
            continue

        for evidence in annotation["evidence"]:
            if not evidence or not evidence.strip():
                continue
            if evidence.strip().upper().startswith("FLOAT SELECTED"):
                continue
            evidence_list.append(evidence.strip())

    return list(dict.fromkeys(evidence_list))


evaluation_questions = []

for paper in eval_papers:
    qas = paper["qas"]
    n_questions = len(qas["question"])

    if MAX_QUESTIONS_PER_PAPER is not None:
        n_questions = min(n_questions, MAX_QUESTIONS_PER_PAPER)

    for i in range(n_questions):
        evidence = extract_evidence(paper, i)
        if not evidence:
            continue

        evaluation_questions.append({
            "paper_id": paper["id"],
            "title": paper["title"],
            "question_id": qas["question_id"][i],
            "question": qas["question"][i],
            "evidence": evidence,
        })

if EVAL_QUESTION_LIMIT is not None:
    evaluation_questions = evaluation_questions[:EVAL_QUESTION_LIMIT]

questions_df = pd.DataFrame(evaluation_questions)
print("Questions with textual evidence:", len(evaluation_questions))
display(questions_df.head())

Questions with textual evidence: 83


,paper_id,title,question_id,question,evidence
0,1912.01214,Cross-lingual Pre-training Based Transfer for ...,b6f15fb6279b82e34a5bf4828b7b5ddabfdf1d54,which multilingual approaches do they compare ...,[Table TABREF19 and TABREF26 report zero-shot ...
1,1912.01214,Cross-lingual Pre-training Based Transfer for ...,f5e6f43454332e0521a778db0b769481e23e7682,what are the pivot-based baselines?,[Table TABREF19 and TABREF26 report zero-shot ...
2,1912.01214,Cross-lingual Pre-training Based Transfer for ...,9a05a5f4351db75da371f7ac12eb0b03607c4b87,which datasets did they experiment with?,[We evaluate our cross-lingual pre-training ba...
3,1912.01214,Cross-lingual Pre-training Based Transfer for ...,5eda469a8a77f028d0c5f1acd296111085614537,what language pairs are explored?,"[For MultiUN corpus, we use four languages: En..."
4,1810.08699,pioNER: Datasets and Baselines for Armenian Na...,18c5d366b1da8447b5404eab71f4cc658ba12e6f,what ner models were evaluated?,[In this section we describe a number of exper...


In [ ]:
def split_into_chunks(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be >= 0 and smaller than chunk_size")

    words = text.split()
    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk_words = words[start:start + chunk_size]
        if not chunk_words:
            break
        chunks.append(" ".join(chunk_words))
        if start + chunk_size >= len(words):
            break

    return chunks


def section_aware_chunking(paper, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    full_text = paper["full_text"]

    for section_idx, section_name in enumerate(full_text["section_name"]):
        paragraphs = full_text["paragraphs"][section_idx]
        section_text = "\\n\\n".join(
            p.strip() for p in paragraphs if p and p.strip()
        )

        if not section_text:
            continue

        parts = split_into_chunks(
            section_text,
            chunk_size=chunk_size,
            overlap=overlap,
        )

        for chunk_idx, text in enumerate(parts):
            chunks.append({
                "paper_id": paper["id"],
                "title": paper["title"],
                "section": section_name,
                "chunk_index": chunk_idx,
                "text": text,
            })

    return chunks


# Build only section-aware chunks.
section_aware_chunks = []

for paper in eval_papers:
    section_aware_chunks.extend(section_aware_chunking(paper))

chunks_by_strategy = {
    "section_aware": section_aware_chunks
}

print(f"section_aware: {len(section_aware_chunks):,} chunks")

In [ ]:
model = SentenceTransformer(MODEL_NAME)
print("Loaded:", MODEL_NAME)
print("Device:", model.device)


def build_paper_indexes(chunks, model):
    grouped = defaultdict(list)

    for chunk in chunks:
        grouped[chunk["paper_id"]].append(chunk)

    indexes = {}

    for paper_id, paper_chunks in grouped.items():
        texts = [
            (
                f"Title: {chunk['title']}\n"
                f"Section: {chunk['section']}\n"
                f"Content: {chunk['text']}"
            )
            for chunk in paper_chunks
        ]

        embeddings = model.encode(
            texts,
            batch_size=32,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype("float32")

        index = faiss.IndexFlatIP(embeddings.shape[1])
        index.add(embeddings)

        indexes[paper_id] = {
            "index": index,
            "chunks": paper_chunks,
        }

    return indexes


indexes_by_strategy = {}
for strategy, chunks in chunks_by_strategy.items():
    print("Building:", strategy)
    indexes_by_strategy[strategy] = build_paper_indexes(chunks, model)

print("All FAISS indexes are ready.")

In [ ]:
def retrieve_chunks(question, paper_id, paper_indexes, top_k=RERANK_TOP_K):
    paper_data = paper_indexes.get(paper_id)
    if paper_data is None:
        return []

    index = paper_data["index"]
    chunks = paper_data["chunks"]

    if index.ntotal == 0:
        return []

    query_embedding = model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype("float32")

    scores, indices = index.search(
        query_embedding,
        min(top_k, index.ntotal),
    )

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue
        results.append({
            **chunks[int(idx)],
            "score": float(score),
        })

    return results


def normalize_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text.strip()


def chunk_matches_evidence(chunk_text, evidence_list):
    chunk_norm = normalize_text(chunk_text)

    if len(chunk_norm) < 80:
        return False

    for evidence in evidence_list:
        evidence_norm = normalize_text(evidence)
        if not evidence_norm:
            continue
        if evidence_norm in chunk_norm:
            return True
        if chunk_norm in evidence_norm:
            return True

    return False

In [ ]:
def _apply_scores(chunks, scores):
    """Attach rerank scores and return chunks sorted from highest to lowest."""
    scored_chunks = []

    for chunk, score in zip(chunks, scores):
        item = dict(chunk)
        item["rerank_score"] = float(score)
        scored_chunks.append(item)

    scored_chunks.sort(
        key=lambda item: item["rerank_score"],
        reverse=True,
    )

    for rank, chunk in enumerate(scored_chunks, start=1):
        chunk["rerank_rank"] = rank

    return scored_chunks

In [ ]:
class CrossEncoderReRanker:
    def __init__(self, model_name=CROSS_ENCODER_MODEL):
        self.model_name = model_name
        self.model = CrossEncoder(model_name)

    def rerank(self, question, chunks):
        if not chunks:
            return []

        pairs = [
            (question, chunk["text"])
            for chunk in chunks
        ]
        scores = self.model.predict(pairs)
        return _apply_scores(chunks, scores)


cross_encoder_reranker = CrossEncoderReRanker()
print("Cross-Encoder loaded:", CROSS_ENCODER_MODEL)

In [ ]:
from rag.chat_model import EuriChatModel


class LLMReRanker:
    def __init__(self):
        self.model = EuriChatModel()

    def rerank(self, question, chunks):
        if not chunks:
            return []

        numbered_chunks = "\n\n".join(
            f"CHUNK {i}: {chunk['text']}"
            for i, chunk in enumerate(chunks, start=1)
        )

        prompt = f"""
You are a retrieval reranking system.
Rank the following chunks by how well they answer the question.
Use only the information in the chunks.

QUESTION:
{question}

CHUNKS:
{numbered_chunks}

Return ONLY the chunk numbers in best-to-worst order, separated by commas.
Include each chunk number exactly once.
"""

        response = self.model.invoke(prompt)
        response_text = (
            response.content
            if hasattr(response, "content")
            else str(response)
        )

        ranked_ids = [
            int(value)
            for value in re.findall(r"\b\d+\b", response_text)
            if 1 <= int(value) <= len(chunks)
        ]

        # Remove duplicates while preserving the LLM's ordering.
        ranked_ids = list(dict.fromkeys(ranked_ids))

        # Add any omitted chunks at the end in their original retrieval order.
        ranked_ids.extend(
            i for i in range(1, len(chunks) + 1)
            if i not in ranked_ids
        )

        reordered = [dict(chunks[i - 1]) for i in ranked_ids]
        scores = list(
            range(len(reordered), 0, -1)
        )

        return _apply_scores(reordered, scores)


llm_reranker = LLMReRanker()
print("LLM reranker initialized.")

In [ ]:
rerankers = {
    "no_reranker": None,
    "cross_encoder": cross_encoder_reranker,
    "llm": llm_reranker,
}

reranking_rows = []
reranking_results = {}

for strategy, paper_indexes in indexes_by_strategy.items():
    print(f"\nRunning strategy: {strategy}")

    for question_number, q in enumerate(evaluation_questions, start=1):
        retrieved_chunks = retrieve_chunks(
            q["question"],
            q["paper_id"],
            paper_indexes,
            RERANK_TOP_K,
        )

        # Preserve the original vector-search rank for analysis.
        for rank, chunk in enumerate(retrieved_chunks, start=1):
            chunk["vector_rank"] = rank

        for method_name, reranker in rerankers.items():
            start_time = time.time()

            if reranker is None:
                # No Reranker control: same retrieved chunks, original order.
                ranked_chunks = [dict(chunk) for chunk in retrieved_chunks]
            else:
                ranked_chunks = reranker.rerank(
                    q["question"],
                    retrieved_chunks,
                )

            elapsed = time.time() - start_time
            top_chunks = ranked_chunks[:TOP_N]

            reranking_results[
                (strategy, q["question_id"], method_name)
            ] = top_chunks

            relevance = [
                chunk_matches_evidence(chunk["text"], q["evidence"])
                for chunk in top_chunks
            ]

            # Count all relevant chunks in this paper's chunk collection.
            all_chunks = paper_indexes[q["paper_id"]]["chunks"]
            total_relevant = sum(
                chunk_matches_evidence(chunk["text"], q["evidence"])
                for chunk in all_chunks
            )

            relevant_ranks = [
                rank for rank, is_relevant in enumerate(relevance, start=1)
                if is_relevant
            ]

            recall_at_5 = (
                sum(relevance) / total_relevant
                if total_relevant else np.nan
            )
            precision_at_5 = sum(relevance) / TOP_N
            mrr = 1 / relevant_ranks[0] if relevant_ranks else 0.0

            reranking_rows.append({
                "strategy": strategy,
                "method": method_name,
                "paper_id": q["paper_id"],
                "question_id": q["question_id"],
                "num_retrieved": len(retrieved_chunks),
                "num_relevant_in_top5": sum(relevance),
                "recall_at_5": recall_at_5,
                "precision_at_5": precision_at_5,
                "mrr": mrr,
                "rerank_time_seconds": elapsed,
                "evaluated": total_relevant > 0,
            })

        if question_number % 10 == 0:
            print(f"  Finished {question_number}/{len(evaluation_questions)} questions")

reranking_results_df = pd.DataFrame(reranking_rows)
print("\nReranking evaluation finished.")
display(reranking_results_df.head())

In [ ]:
valid_results = reranking_results_df[
    reranking_results_df["evaluated"]
].copy()

reranking_summary_df = (
    valid_results
    .groupby(["strategy", "method"])
    .agg(
        evaluated_questions=("question_id", "count"),
        recall_at_5=("recall_at_5", "mean"),
        precision_at_5=("precision_at_5", "mean"),
        mrr=("mrr", "mean"),
        avg_rerank_time_seconds=("rerank_time_seconds", "mean"),
    )
    .reset_index()
    .sort_values(["strategy", "recall_at_5"], ascending=[True, False])
)

display(reranking_summary_df.round(4))

In [ ]:
for metric in ["recall_at_5", "precision_at_5", "mrr"]:
    pivot = reranking_summary_df.pivot(
        index="strategy",
        columns="method",
        values=metric,
    )

    ax = pivot.plot(
        kind="bar",
        figsize=(10, 5),
        title=f"{metric.replace('_', ' ').title()} by chunking strategy and reranker",
    )
    ax.set_ylabel("Score")
    ax.set_xlabel("Chunking strategy")
    ax.set_ylim(0, 1)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:

def _apply_scores(chunks: list[dict], scores: list[float]) -> list[dict]:
    """
    Adds reranking scores to chunks, sorts them, and assigns new ranks.
    """

    # Add the reranking score to each chunk
    scored = [
        {**chunk, "rerank_score": float(score)}
        for chunk, score in zip(chunks, scores)
    ]

    # Sort by reranking score (highest first)
    scored.sort(
        key=lambda c: c["rerank_score"],
        reverse=True
    )

    # Assign new ranks
    for new_rank, chunk in enumerate(scored, start=1):
        chunk["rerank_rank"] = new_rank

    return scored


In [ ]:

## Reranker 1: Cross-Encoder

The Cross-Encoder evaluates the question and each retrieved chunk together.

Unlike the original vector search, which compares embeddings, the Cross-Encoder reads the question and chunk simultaneously.

It produces a relevance score for each chunk.

Higher scores indicate that a chunk is more relevant to the question.

The model used is:
`cross-encoder/ms-marco-MiniLM-L-6-v2`


In [ ]:

class CrossEncoderReRanker:

    name = "cross-encoder"

    def __init__(
        self,
        model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    ):
        import torch
        from transformers import (
            AutoTokenizer,
            AutoModelForSequenceClassification
        )

        self.torch = torch

        # Load tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        # Load pretrained Cross-Encoder model
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name
        )

        # Switch model to evaluation mode
        self.model.eval()

    def __call__(self, query: str, chunks: list[dict]) -> list[dict]:

        # Extract text from each chunk
        texts = [chunk["text"] for chunk in chunks]

        # Combine the query with each chunk
        inputs = self.tokenizer(
            [query] * len(texts),
            texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        )

        # Generate relevance scores
        with self.torch.no_grad():

            scores = (
                self.model(**inputs)
                .logits
                .squeeze(-1)
                .tolist()
            )

        # Apply scores and reorder chunks
        return _apply_scores(chunks, scores)


In [ ]:

## Reranker 2: LLM

The LLM reranker sends all retrieved chunks to a general-purpose language model.

The model is instructed to:
- Read the user's question.
- Evaluate all retrieved chunks.
- Rank the chunks from most useful to least useful.
- Return only the chunk numbers.

The response is converted into numerical scores, which are passed to `_apply_scores()`.

This allows us to compare the LLM reranker with the Cross-Encoder using the same output structure.


In [ ]:

class LLMReRanker:

    name = "llm"

    _SYSTEM_PROMPT = (
        "You are a search result reranker. "
        "You reply with only chunk numbers"
    )

    _PROMPT = """Question: {query}

Below are {n} text chunks extracted from PDF documents.
Some text may be scrambled by the PDF extraction.

Rank all {n} chunks from most useful to least useful for answering the question.

A chunk containing the specific details asked for is more useful
than one that only mentions the topic in general.

{chunks}

Reply with only the chunk numbers in order, separated by commas,
for example: 4, 1, 7, ..."""

    def __init__(self):

        self.chat_model = EuriChatModel()

    def __call__(self, query: str, chunks: list[dict]) -> list[dict]:

        # Number all chunks so the LLM can identify them
        numbered = "\n\n".join(
            f"[{i}] {chunk['text']}"
            for i, chunk in enumerate(chunks, start=1)
        )

        # Create the prompt
        prompt = self._PROMPT.format(
            query=query,
            n=len(chunks),
            chunks=numbered
        )

        # Send the prompt to the LLM
        reply = self.chat_model(
            self._SYSTEM_PROMPT,
            prompt
        )

        # Extract chunk numbers from the LLM response
        order = [
            int(n)
            for n in re.findall(r"\d+", reply)
        ]

        # Convert ranking positions into scores
        scores = []

        for label in range(1, len(chunks) + 1):

            if label in order:
                scores.append(
                    len(chunks) - order.index(label)
                )

            else:
                scores.append(0)

        # Apply scores and reorder chunks
        return _apply_scores(chunks, scores)
